# Lucid Classifier Inspection

This notebook inspects the Lucid-style classifier used to predict sharing-score labels from solo-profile features.


In [36]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

FEATURES_LUCID_FAITHFUL = [
    "peak_memory_mib",
    "memory_fraction",
    "horus_gpu_util_mean",
    "avg_drama",
    "amp_enabled",
]

FEATURES_EXTENDED = [
    *FEATURES_LUCID_FAITHFUL,
    "avg_smact",
    "avg_smocc",
    "horus_gpu_util_p95",
    "horus_gpu_util_max",
]

feature_table_path = Path("../results/lucid_feature_table.csv")
df = pd.read_csv(feature_table_path)
df.head()


,spec_name,spec_path,spec_key,peak_memory_mib,memory_fraction,horus_gpu_util_mean,horus_gpu_util_p95,horus_gpu_util_max,avg_smact,avg_smocc,...,representative_spec_path,lucid_mean_normalized_speed,lucid_std_normalized_speed,lucid_min_normalized_speed,lucid_max_normalized_speed,lucid_num_pair_observations,lucid_class,lucid_ss,lucid_label_source,lucid_label_usable
0,efficientnet_cifar100_bs128_20e_1gpu.yaml,evaluation/workloads/training/specs/yaml/effic...,efficientnet_cifar100_bs128_20e_1gpu.yaml,860,0.020996,37.251852,44.0,56.0,0.226215,0.143911,...,evaluation/workloads/training/specs/yaml_thres...,0.944974,0.108199,0.574695,1.0,15,medium,1,measured_pairwise,True
1,mobilenet_cifar100_bs128_20e_1gpu.yaml,evaluation/workloads/training/specs/yaml/mobil...,mobilenet_cifar100_bs128_20e_1gpu.yaml,634,0.015479,32.615385,40.0,46.0,0.157754,0.090031,...,evaluation/workloads/training/specs/yaml_thres...,0.892847,0.145344,0.486047,1.0,17,medium,1,measured_pairwise,True
2,mobilenet_cifar100_bs64_20e_1gpu.yaml,evaluation/workloads/training/specs/yaml/mobil...,mobilenet_cifar100_bs64_20e_1gpu.yaml,602,0.014697,30.969231,39.0,46.0,0.118638,0.061992,...,evaluation/workloads/training/specs/yaml_thres...,0.999147,0.001907,0.994884,1.0,6,tiny,0,measured_pairwise,True
3,resnet18_cifar100_bs64_20e_1gpu.yaml,evaluation/workloads/training/specs/yaml/resne...,resnet18_cifar100_bs64_20e_1gpu.yaml,798,0.019482,34.338346,42.4,60.0,0.166519,0.062098,...,evaluation/workloads/training/specs/yaml_thres...,0.905338,0.138343,0.521719,1.0,14,medium,1,measured_pairwise,True
4,resnet34_cifar100_bs128_20e_1gpu.yaml,evaluation/workloads/training/specs/yaml/resne...,resnet34_cifar100_bs128_20e_1gpu.yaml,1076,0.026270,36.476562,50.0,65.0,0.201719,0.079375,...,evaluation/workloads/training/specs/yaml_thres...,0.816509,0.178654,0.428834,1.0,16,jumbo,2,measured_pairwise,True


## Label coverage

In [37]:
print("Rows:", len(df))
print("\nUsable label counts:")
print(df["lucid_label_usable"].value_counts(dropna=False))

print("\nClass counts:")
print(df["lucid_class"].value_counts(dropna=False))

df[[
    "spec_key",
    "lucid_num_pair_observations",
    "lucid_label_usable",
    "lucid_class",
    "lucid_ss",
]].sort_values(["lucid_label_usable", "lucid_num_pair_observations"], ascending=[True, True])


Rows: 26

Usable label counts:
lucid_label_usable
True     25
False     1
Name: count, dtype: int64

Class counts:
lucid_class
jumbo     10
medium     9
tiny       7
Name: count, dtype: int64


,spec_key,lucid_num_pair_observations,lucid_label_usable,lucid_class,lucid_ss
8,efficientnet_cifar100_bs64_25e_1gpu.yaml,2,False,tiny,0
6,efficientnet_cifar100_bs32_10e_1gpu.yaml,4,True,tiny,0
11,resnet34_cifar100_bs32_15e_1gpu.yaml,4,True,tiny,0
7,efficientnet_cifar100_bs32_12e_1gpu.yaml,5,True,tiny,0
9,mobilenet_cifar100_bs32_15e_1gpu.yaml,5,True,medium,1
10,resnet18_cifar100_bs32_25e_1gpu.yaml,5,True,tiny,0
2,mobilenet_cifar100_bs64_20e_1gpu.yaml,6,True,tiny,0
5,resnet34_cifar100_bs64_20e_1gpu.yaml,6,True,tiny,0
14,llama3_width_8layer_wiki_bs1_1gpu.yaml,9,True,medium,1
23,xception_imagenet_bs128_1gpu.yaml,10,True,medium,1


## Train/evaluate classifier

In [38]:
def train_and_report(feature_cols, seed=42):
    train = df[df["lucid_label_usable"] == True].copy()
    X = train[feature_cols]
    y = train["lucid_ss"].astype(int)

    model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=200,
                    random_state=seed,
                    class_weight="balanced",
                    min_samples_leaf=1,
                ),
            ),
        ]
    )

    model.fit(X, y)

    if len(train) >= 3:
        pred = cross_val_predict(model, X, y, cv=LeaveOneOut())
        print("Leave-one-out report")
        print(classification_report(y, pred, zero_division=0))
        print("Confusion matrix [0=tiny, 1=medium, 2=jumbo]")
        print(confusion_matrix(y, pred, labels=[0, 1, 2]))

    importances = model.named_steps["clf"].feature_importances_
    imp = pd.DataFrame({"feature": feature_cols, "importance": importances})
    imp = imp.sort_values("importance", ascending=False)

    return model, imp

model_lucid, imp_lucid = train_and_report(FEATURES_LUCID_FAITHFUL)
imp_lucid


Leave-one-out report
              precision    recall  f1-score   support

           0       0.50      0.83      0.62         6
           1       0.50      0.33      0.40         9
           2       0.78      0.70      0.74        10

    accuracy                           0.60        25
   macro avg       0.59      0.62      0.59        25
weighted avg       0.61      0.60      0.59        25

Confusion matrix [0=tiny, 1=medium, 2=jumbo]
[[5 1 0]
 [4 3 2]
 [1 2 7]]


,feature,importance
0,peak_memory_mib,0.295377
1,memory_fraction,0.268129
3,avg_drama,0.221539
2,horus_gpu_util_mean,0.214954
4,amp_enabled,0.000000


## Extended feature set

In [39]:
model_ext, imp_ext = train_and_report(FEATURES_EXTENDED)
imp_ext


Leave-one-out report
              precision    recall  f1-score   support

           0       0.44      0.67      0.53         6
           1       0.75      0.33      0.46         9
           2       0.75      0.90      0.82        10

    accuracy                           0.64        25
   macro avg       0.65      0.63      0.60        25
weighted avg       0.68      0.64      0.62        25

Confusion matrix [0=tiny, 1=medium, 2=jumbo]
[[4 1 1]
 [4 3 2]
 [1 0 9]]


,feature,importance
0,peak_memory_mib,0.166713
1,memory_fraction,0.152400
5,avg_smact,0.139663
6,avg_smocc,0.135987
2,horus_gpu_util_mean,0.112916
7,horus_gpu_util_p95,0.111354
3,avg_drama,0.109012
8,horus_gpu_util_max,0.071955
4,amp_enabled,0.000000


## Compare measured and predicted labels

In [40]:
def predict_table(model, feature_cols, source_name):
    out = df.copy()
    pred = model.predict(out[feature_cols])
    proba = model.predict_proba(out[feature_cols])
    classes = list(model.named_steps["clf"].classes_)

    ss_to_class = {0: "tiny", 1: "medium", 2: "jumbo"}

    out[f"pred_ss_{source_name}"] = pred
    out[f"pred_class_{source_name}"] = [ss_to_class[int(x)] for x in pred]

    for i, cls in enumerate(classes):
        out[f"pred_proba_ss{int(cls)}_{source_name}"] = proba[:, i]

    return out

pred_lucid = predict_table(model_lucid, FEATURES_LUCID_FAITHFUL, "lucid")
pred_lucid[[
    "spec_key",
    "lucid_label_usable",
    "lucid_class",
    "lucid_ss",
    "pred_class_lucid",
    "pred_ss_lucid",
    "pred_proba_ss0_lucid",
    "pred_proba_ss1_lucid",
    "pred_proba_ss2_lucid",
]].sort_values("spec_key")


,spec_key,lucid_label_usable,lucid_class,lucid_ss,pred_class_lucid,pred_ss_lucid,pred_proba_ss0_lucid,pred_proba_ss1_lucid,pred_proba_ss2_lucid
0,efficientnet_cifar100_bs128_20e_1gpu.yaml,True,medium,1,medium,1,0.245,0.745,0.010
6,efficientnet_cifar100_bs32_10e_1gpu.yaml,True,tiny,0,tiny,0,0.890,0.100,0.010
7,efficientnet_cifar100_bs32_12e_1gpu.yaml,True,tiny,0,tiny,0,0.885,0.115,0.000
8,efficientnet_cifar100_bs64_25e_1gpu.yaml,False,tiny,0,tiny,0,0.905,0.085,0.010
12,efficientnet_imagenet_bs128_1gpu.yaml,True,jumbo,2,jumbo,2,0.000,0.220,0.780
13,efficientnet_imagenet_bs64_1gpu.yaml,True,jumbo,2,jumbo,2,0.000,0.005,0.995
14,llama3_width_8layer_wiki_bs1_1gpu.yaml,True,medium,1,medium,1,0.000,0.870,0.130
1,mobilenet_cifar100_bs128_20e_1gpu.yaml,True,medium,1,medium,1,0.370,0.630,0.000
9,mobilenet_cifar100_bs32_15e_1gpu.yaml,True,medium,1,medium,1,0.245,0.755,0.000
2,mobilenet_cifar100_bs64_20e_1gpu.yaml,True,tiny,0,tiny,0,0.690,0.310,0.000


## Export notebook predictions

In [41]:
output = Path("../results/lucid_classifier_predictions_notebook.csv")
pred_lucid.to_csv(output, index=False)
print(output)


../results/lucid_classifier_predictions_notebook.csv


In [42]:
from pathlib import Path
import yaml
import pandas as pd

ALL_SPEC_DIR = Path("../../workloads/training/specs/yaml")
OUTPUT_ALL_SPECS = Path("../results/lucid_predictions_all_specs.csv")

def canonical_spec_key(path):
    name = Path(path).name
    stem = Path(name).stem

    suffixes = [
        "_maxbatches1200",
        "_maxbatches600",
        "_maxbatches",
        "_maxsteps2000",
    ]
    for suffix in suffixes:
        if stem.endswith(suffix):
            stem = stem[: -len(suffix)]
            break

    if not stem.endswith("_1gpu"):
        stem = f"{stem}_1gpu"

    return f"{stem}.yaml"

def command_has_amp(command):
    return int("--amp" in str(command).split())

def build_feature_row(spec_path, gpu_capacity_mib=40960):
    data = yaml.safe_load(spec_path.read_text()) or {}
    profile = data.get("profile", {}) or {}
    job = data.get("job", {}) or {}
    command = str(job.get("command", ""))

    peak_memory_mib = profile.get("peak_memory_mib")

    return {
        "spec_name": spec_path.name,
        "spec_path": str(spec_path),
        "spec_key": canonical_spec_key(spec_path),
        "peak_memory_mib": peak_memory_mib,
        "memory_fraction": None if peak_memory_mib is None else float(peak_memory_mib) / gpu_capacity_mib,
        "horus_gpu_util_mean": profile.get("horus_gpu_util_mean"),
        "horus_gpu_util_p95": profile.get("horus_gpu_util_p95"),
        "horus_gpu_util_max": profile.get("horus_gpu_util_max"),
        "avg_smact": profile.get("avg_smact"),
        "avg_smocc": profile.get("avg_smocc"),
        "avg_drama": profile.get("avg_drama"),
        "amp_enabled": command_has_amp(command),
    }

all_specs = sorted(ALL_SPEC_DIR.glob("*.yaml"))
all_features = pd.DataFrame([build_feature_row(p) for p in all_specs])

pred_ss = model_lucid.predict(all_features[FEATURES_LUCID_FAITHFUL])
pred_proba = model_lucid.predict_proba(all_features[FEATURES_LUCID_FAITHFUL])
classes = list(model_lucid.named_steps["clf"].classes_)

ss_to_class = {0: "tiny", 1: "medium", 2: "jumbo"}

all_predictions = all_features.copy()
all_predictions["lucid_pred_ss"] = pred_ss
all_predictions["lucid_pred_class"] = [ss_to_class[int(x)] for x in pred_ss]
all_predictions["lucid_pred_source"] = "predicted_classifier_lucid_faithful"

for i, cls in enumerate(classes):
    all_predictions[f"lucid_pred_proba_ss{int(cls)}"] = pred_proba[:, i]

all_predictions.to_csv(OUTPUT_ALL_SPECS, index=False)

print("rows:", len(all_predictions))
print("wrote:", OUTPUT_ALL_SPECS)
print(all_predictions[[
    "spec_name",
    "lucid_pred_class",
    "lucid_pred_ss",
    "lucid_pred_proba_ss0",
    "lucid_pred_proba_ss1",
    "lucid_pred_proba_ss2",
]].to_string(index=False))

rows: 53
wrote: ../results/lucid_predictions_all_specs.csv
                                spec_name lucid_pred_class  lucid_pred_ss  lucid_pred_proba_ss0  lucid_pred_proba_ss1  lucid_pred_proba_ss2
            bert_base_wiki_bs32_1gpu.yaml           medium              1                 0.040                 0.790                 0.170
            bert_large_wiki_bs8_1gpu.yaml            jumbo              2                 0.040                 0.300                 0.660
            dlrm_criteo_bs32768_1gpu.yaml           medium              1                 0.275                 0.370                 0.355
efficientnet_cifar100_bs128_20e_1gpu.yaml           medium              1                 0.245                 0.745                 0.010
efficientnet_cifar100_bs128_50e_1gpu.yaml           medium              1                 0.460                 0.530                 0.010
 efficientnet_cifar100_bs32_20e_1gpu.yaml             tiny              0                 0.890      

In [43]:
proba_cols = ["lucid_pred_proba_ss0", "lucid_pred_proba_ss1", "lucid_pred_proba_ss2"]
all_predictions["max_proba"] = all_predictions[proba_cols].max(axis=1)

all_predictions.sort_values("max_proba")[[
    "spec_name",
    "lucid_pred_class",
    "lucid_pred_ss",
    "max_proba",
    *proba_cols,
]]

,spec_name,lucid_pred_class,lucid_pred_ss,max_proba,lucid_pred_proba_ss0,lucid_pred_proba_ss1,lucid_pred_proba_ss2
19,mnist_bs32_1gpu.yaml,medium,1,0.350,0.300,0.350,0.350
2,dlrm_criteo_bs32768_1gpu.yaml,medium,1,0.370,0.275,0.370,0.355
30,resnet18_cifar100_bs128_50e_1gpu.yaml,medium,1,0.455,0.425,0.455,0.120
29,resnet18_cifar100_bs128_20e_1gpu.yaml,medium,1,0.485,0.475,0.485,0.040
52,xlnet_large_cased_wiki_bs4_2gpu.yaml,jumbo,2,0.525,0.000,0.475,0.525
4,efficientnet_cifar100_bs128_50e_1gpu.yaml,medium,1,0.530,0.460,0.530,0.010
34,resnet18_cifar100_bs64_50e_1gpu.yaml,medium,1,0.540,0.455,0.540,0.005
16,inception_imagenet_bs64_1gpu.yaml,jumbo,2,0.570,0.000,0.430,0.570
40,resnet34_cifar100_bs64_50e_1gpu.yaml,tiny,0,0.590,0.590,0.350,0.060
25,mobilenet_cifar100_bs64_50e_1gpu.yaml,tiny,0,0.620,0.620,0.380,0.000


## Tiny-class safeguards
The classifier has weak Tiny support, so we do not blindly trust classifier outputs for low-resource jobs.

In [44]:
train = df[df["lucid_label_usable"] == True].copy()

print("Training class counts:")
print(train["lucid_class"].value_counts().to_string())

tiny_train = train[train["lucid_ss"] == 0]
print("\nTiny training examples:")
display(tiny_train[[
    "spec_key",
    "peak_memory_mib",
    "memory_fraction",
    "horus_gpu_util_mean",
    "avg_smact",
    "avg_smocc",
    "avg_drama",
    "lucid_mean_normalized_speed",
    "lucid_num_pair_observations",
]])


Training class counts:
lucid_class
jumbo     10
medium     9
tiny       6

Tiny training examples:


,spec_key,peak_memory_mib,memory_fraction,horus_gpu_util_mean,avg_smact,avg_smocc,avg_drama,lucid_mean_normalized_speed,lucid_num_pair_observations
2,mobilenet_cifar100_bs64_20e_1gpu.yaml,602,0.014697,30.969231,0.118638,0.061992,0.003038,0.999147,6
5,resnet34_cifar100_bs64_20e_1gpu.yaml,1018,0.024854,36.308271,0.179992,0.072256,0.060128,0.988568,6
6,efficientnet_cifar100_bs32_10e_1gpu.yaml,668,0.016309,36.830769,0.159577,0.087654,0.008246,0.982451,4
7,efficientnet_cifar100_bs32_12e_1gpu.yaml,668,0.016309,34.931298,0.152466,0.084298,0.007962,0.998598,5
10,resnet18_cifar100_bs32_25e_1gpu.yaml,790,0.019287,35.863636,0.173242,0.063311,0.049705,0.994941,5
11,resnet34_cifar100_bs32_15e_1gpu.yaml,1004,0.024512,36.492424,0.181167,0.071697,0.057992,0.967820,4


In [45]:
# Conservative tiny-like heuristic:
# A workload is "tiny-like" if it falls below the measured Tiny envelope
# for memory and GPU utilization.

tiny_memory_max = tiny_train["peak_memory_mib"].max()
tiny_util_max = tiny_train["horus_gpu_util_mean"].max()

print("Tiny envelope:")
print("max peak_memory_mib:", tiny_memory_max)
print("max horus_gpu_util_mean:", tiny_util_max)

all_predictions["tiny_like_by_profile"] = (
    (all_predictions["peak_memory_mib"] <= tiny_memory_max)
    & (all_predictions["horus_gpu_util_mean"] <= tiny_util_max)
)

tiny_conflicts = all_predictions[
    (all_predictions["tiny_like_by_profile"])
    & (all_predictions["lucid_pred_ss"] != 0)
].copy()

print("Tiny-like workloads predicted as non-tiny:", len(tiny_conflicts))

display(tiny_conflicts[[
    "spec_name",
    "peak_memory_mib",
    "memory_fraction",
    "horus_gpu_util_mean",
    "avg_smact",
    "avg_smocc",
    "avg_drama",
    "lucid_pred_class",
    "lucid_pred_ss",
    "lucid_pred_proba_ss0",
    "lucid_pred_proba_ss1",
    "lucid_pred_proba_ss2",
]])

Tiny envelope:
max peak_memory_mib: 1018
max horus_gpu_util_mean: 36.83076923076923
Tiny-like workloads predicted as non-tiny: 9


,spec_name,peak_memory_mib,memory_fraction,horus_gpu_util_mean,avg_smact,avg_smocc,avg_drama,lucid_pred_class,lucid_pred_ss,lucid_pred_proba_ss0,lucid_pred_proba_ss1,lucid_pred_proba_ss2
4,efficientnet_cifar100_bs128_50e_1gpu.yaml,860,0.020996,36.222222,0.224467,0.141696,0.019807,medium,1,0.460,0.530,0.010
20,mobilenet_cifar100_bs128_20e_1gpu.yaml,634,0.015479,32.615385,0.157754,0.090031,0.005069,medium,1,0.370,0.630,0.000
21,mobilenet_cifar100_bs128_50e_1gpu.yaml,634,0.015479,31.992366,0.154870,0.086794,0.004511,medium,1,0.340,0.660,0.000
22,mobilenet_cifar100_bs32_20e_1gpu.yaml,592,0.014453,24.348837,0.071473,0.035566,0.001744,medium,1,0.245,0.755,0.000
23,mobilenet_cifar100_bs32_50e_1gpu.yaml,592,0.014453,29.546154,0.087785,0.043662,0.001900,medium,1,0.245,0.755,0.000
29,resnet18_cifar100_bs128_20e_1gpu.yaml,852,0.020801,34.329412,0.180988,0.067953,0.062800,medium,1,0.475,0.485,0.040
30,resnet18_cifar100_bs128_50e_1gpu.yaml,852,0.020801,33.948148,0.182963,0.068422,0.064119,medium,1,0.425,0.455,0.120
33,resnet18_cifar100_bs64_20e_1gpu.yaml,798,0.019482,34.338346,0.166519,0.062098,0.049241,medium,1,0.340,0.655,0.005
34,resnet18_cifar100_bs64_50e_1gpu.yaml,798,0.019482,34.601504,0.168286,0.063120,0.051835,medium,1,0.455,0.540,0.005


In [46]:
all_predictions["lucid_final_ss"] = all_predictions["lucid_pred_ss"]
all_predictions["lucid_final_class"] = all_predictions["lucid_pred_class"]
all_predictions["lucid_final_source"] = all_predictions["lucid_pred_source"]

# Apply conservative Tiny safeguard.
tiny_conflict_mask = (
    all_predictions["tiny_like_by_profile"]
    & (all_predictions["lucid_pred_ss"] != 0)
)

all_predictions.loc[tiny_conflict_mask, "lucid_final_ss"] = 0
all_predictions.loc[tiny_conflict_mask, "lucid_final_class"] = "tiny"
all_predictions.loc[tiny_conflict_mask, "lucid_final_source"] = (
    "tiny_profile_safeguard_after_classifier"
)

print("Final class counts after safeguard:")
print(all_predictions["lucid_final_class"].value_counts().to_string())

display(all_predictions[[
    "spec_name",
    "peak_memory_mib",
    "horus_gpu_util_mean",
    "tiny_like_by_profile",
    "lucid_pred_class",
    "lucid_final_class",
    "lucid_final_source",
]].sort_values(["lucid_final_class", "peak_memory_mib"]))

Final class counts after safeguard:
lucid_final_class
tiny      21
jumbo     18
medium    14


,spec_name,peak_memory_mib,horus_gpu_util_mean,tiny_like_by_profile,lucid_pred_class,lucid_final_class,lucid_final_source
35,resnet34_cifar100_bs128_20e_1gpu.yaml,1076,36.476562,False,jumbo,jumbo,predicted_classifier_lucid_faithful
36,resnet34_cifar100_bs128_50e_1gpu.yaml,1076,39.088889,False,jumbo,jumbo,predicted_classifier_lucid_faithful
27,mobilenet_imagenet_bs32_1gpu.yaml,3354,63.614815,False,jumbo,jumbo,predicted_classifier_lucid_faithful
10,efficientnet_imagenet_bs32_1gpu.yaml,3760,62.696296,False,jumbo,jumbo,predicted_classifier_lucid_faithful
42,resnet50_imagenet_bs32_1gpu.yaml,3944,77.425373,False,jumbo,jumbo,predicted_classifier_lucid_faithful
15,inception_imagenet_bs32_1gpu.yaml,5246,75.451852,False,jumbo,jumbo,predicted_classifier_lucid_faithful
44,unet_voc_1gpu.yaml,5636,84.201493,False,jumbo,jumbo,predicted_classifier_lucid_faithful
49,xception_imagenet_bs32_1gpu.yaml,5922,82.395522,False,jumbo,jumbo,predicted_classifier_lucid_faithful
28,mobilenet_imagenet_bs64_1gpu.yaml,6162,64.274074,False,jumbo,jumbo,predicted_classifier_lucid_faithful
46,vgg16_imagenet_bs32_1gpu.yaml,6680,86.134328,False,jumbo,jumbo,predicted_classifier_lucid_faithful


In [47]:
OUTPUT_FINAL_ALL_SPECS = Path("../results/lucid_predictions_all_specs_with_tiny_safeguard.csv")

all_predictions.to_csv(OUTPUT_FINAL_ALL_SPECS, index=False)

print("wrote:", OUTPUT_FINAL_ALL_SPECS)

wrote: ../results/lucid_predictions_all_specs_with_tiny_safeguard.csv


Because the measured Lucid labels contain relatively few Tiny examples, the classifier may be biased against predicting Tiny. To avoid overclassifying low-resource workloads as Medium/Jumbo, we add a conservative post-classification safeguard. If a workload falls inside the measured Tiny envelope for both peak GPU memory and mean GPU utilization, but the classifier predicts a higher sharing score, we mark it as Tiny and record the source as `tiny_profile_safeguard_after_classifier`. This safeguard is intentionally conservative and transparent.

In [48]:
# Additional conservative safeguard:
# If a classifier-only workload has very low memory footprint and very low
# mean GPU utilization, avoid assigning it Jumbo. We cap it at Medium.
low_resource_mask = (
    (all_predictions["peak_memory_mib"] <= 2048)
    & (all_predictions["horus_gpu_util_mean"] <= 20)
    & (all_predictions["lucid_final_ss"] == 2)
)

all_predictions.loc[low_resource_mask, "lucid_final_ss"] = 1
all_predictions.loc[low_resource_mask, "lucid_final_class"] = "medium"
all_predictions.loc[low_resource_mask, "lucid_final_source"] = (
    "low_resource_safeguard_after_classifier"
)

display(all_predictions[low_resource_mask][[
    "spec_name",
    "peak_memory_mib",
    "horus_gpu_util_mean",
    "lucid_pred_class",
    "lucid_final_class",
    "lucid_final_source",
]])

,spec_name,peak_memory_mib,horus_gpu_util_mean,lucid_pred_class,lucid_final_class,lucid_final_source


In [49]:
# Conservative low-resource safeguard:
# The classifier has weak Tiny support, so very-low-resource classifier-only
# workloads should not be promoted to Medium/Jumbo solely by the classifier.
low_resource_tiny_mask = (
    (all_predictions["peak_memory_mib"] <= 2048)
    & (all_predictions["horus_gpu_util_mean"] <= 20)
)

all_predictions.loc[low_resource_tiny_mask, "lucid_final_ss"] = 0
all_predictions.loc[low_resource_tiny_mask, "lucid_final_class"] = "tiny"
all_predictions.loc[low_resource_tiny_mask, "lucid_final_source"] = (
    "low_resource_tiny_safeguard_after_classifier"
)

display(all_predictions[low_resource_tiny_mask][[
    "spec_name",
    "peak_memory_mib",
    "horus_gpu_util_mean",
    "avg_smact",
    "avg_smocc",
    "avg_drama",
    "lucid_pred_class",
    "lucid_final_class",
    "lucid_final_source",
]])

,spec_name,peak_memory_mib,horus_gpu_util_mean,avg_smact,avg_smocc,avg_drama,lucid_pred_class,lucid_final_class,lucid_final_source
2,dlrm_criteo_bs32768_1gpu.yaml,1364,10.477612,0.139478,0.046537,0.027134,medium,tiny,low_resource_tiny_safeguard_after_classifier
19,mnist_bs32_1gpu.yaml,1098,16.285714,0.079077,0.029780,0.012725,medium,tiny,low_resource_tiny_safeguard_after_classifier


In [50]:
# Manual transparent override for known toy/sanity workloads.
manual_tiny_patterns = ["mnist"]

manual_tiny_mask = all_predictions["spec_name"].str.lower().apply(
    lambda name: any(pattern in name for pattern in manual_tiny_patterns)
)

all_predictions.loc[manual_tiny_mask, "lucid_final_ss"] = 0
all_predictions.loc[manual_tiny_mask, "lucid_final_class"] = "tiny"
all_predictions.loc[manual_tiny_mask, "lucid_final_source"] = (
    "manual_tiny_workload_override"
)

display(all_predictions[manual_tiny_mask][[
    "spec_name",
    "peak_memory_mib",
    "horus_gpu_util_mean",
    "lucid_pred_class",
    "lucid_final_class",
    "lucid_final_source",
]])

,spec_name,peak_memory_mib,horus_gpu_util_mean,lucid_pred_class,lucid_final_class,lucid_final_source
19,mnist_bs32_1gpu.yaml,1098,16.285714,medium,tiny,manual_tiny_workload_override


Because the measured Lucid labels contain relatively few Tiny examples, the classifier tends to overestimate sharing scores for very small workloads. We therefore apply two transparent safeguards. First, classifier-only workloads with both low peak GPU memory and low mean GPU utilization are labeled Tiny. Second, known toy/sanity workloads such as MNIST are explicitly marked Tiny. These safeguards are recorded in the label source field and are not hidden as classifier outputs.

In [51]:
OUTPUT_FINAL_ALL_SPECS = Path("../results/lucid_predictions_all_specs_with_tiny_safeguard.csv")
all_predictions.to_csv(OUTPUT_FINAL_ALL_SPECS, index=False)
print("wrote:", OUTPUT_FINAL_ALL_SPECS)

wrote: ../results/lucid_predictions_all_specs_with_tiny_safeguard.csv


## Partner-dependent labels: why static sharing classes are hard

A key limitation of Lucid-style static classes is that the sharing score assigned to a workload is not only determined by its solo profile. It is also shaped by the partner workloads used during pairwise profiling. In the examples below, workloads with similar solo resource footprints receive different sharing classes because their observed slowdowns differ across colocated partners.

This creates a challenge for static classification: a single Tiny/Medium/Jumbo label compresses many possible pairwise interactions into one per-workload class. Such a label can be useful as a baseline, but it is sensitive to the profiling dataset and may not transfer cleanly when the workload mix changes.


In [52]:
cols = [
    "spec_key",
    "lucid_class",
    "lucid_ss",
    "lucid_mean_normalized_speed",
    "lucid_num_pair_observations",
    "peak_memory_mib",
    "memory_fraction",
    "horus_gpu_util_mean",
    "avg_drama",
    "avg_smact",
    "avg_smocc",
]

examples = df[
    df["spec_key"].isin([
        "resnet34_cifar100_bs64_20e_1gpu.yaml",
        "resnet34_cifar100_bs128_20e_1gpu.yaml",
        "mobilenet_cifar100_bs64_20e_1gpu.yaml",
        "mobilenet_cifar100_bs128_20e_1gpu.yaml",
        "efficientnet_cifar100_bs32_10e_1gpu.yaml",
        "efficientnet_cifar100_bs128_20e_1gpu.yaml",
    ])
].copy()

examples[cols].sort_values("peak_memory_mib")

,spec_key,lucid_class,lucid_ss,lucid_mean_normalized_speed,lucid_num_pair_observations,peak_memory_mib,memory_fraction,horus_gpu_util_mean,avg_drama,avg_smact,avg_smocc
2,mobilenet_cifar100_bs64_20e_1gpu.yaml,tiny,0,0.999147,6,602,0.014697,30.969231,0.003038,0.118638,0.061992
1,mobilenet_cifar100_bs128_20e_1gpu.yaml,medium,1,0.892847,17,634,0.015479,32.615385,0.005069,0.157754,0.090031
6,efficientnet_cifar100_bs32_10e_1gpu.yaml,tiny,0,0.982451,4,668,0.016309,36.830769,0.008246,0.159577,0.087654
0,efficientnet_cifar100_bs128_20e_1gpu.yaml,medium,1,0.944974,15,860,0.020996,37.251852,0.020215,0.226215,0.143911
5,resnet34_cifar100_bs64_20e_1gpu.yaml,tiny,0,0.988568,6,1018,0.024854,36.308271,0.060128,0.179992,0.072256
4,resnet34_cifar100_bs128_20e_1gpu.yaml,jumbo,2,0.816509,16,1076,0.026270,36.476562,0.069383,0.201719,0.079375


The table shows cases where solo-profile features are close but the final sharing labels differ. For example, `resnet34_cifar100_bs64` and `resnet34_cifar100_bs128` have similar memory footprint, GPU utilization, and DRAM activity, yet their pairwise-derived labels differ. The difference is explained by the pairwise observations: the harsher label is driven by slowdowns under specific colocated partners rather than by a large difference in solo resource use.


In [53]:
obs = pd.read_csv("../results/lucid_label_observations.csv")
obs = obs[obs["valid_for_label"] == True].copy()

targets = [
    "resnet34_cifar100_bs128_20e_1gpu.yaml",
    "resnet34_cifar100_bs64_20e_1gpu.yaml",
    "mobilenet_cifar100_bs128_20e_1gpu.yaml",
    "mobilenet_cifar100_bs64_20e_1gpu.yaml",
]

obs_cols = [
    "spec_key",
    "partner_spec_key",
    "normalized_speed",
    "partner_runtime_ratio",
    "pairwise_source_csv",
]

for target in targets:
    x = obs[obs["spec_key"] == target].copy()
    print(f"\n== {target}")
    print("count:", len(x))
    print("mean normalized speed:", round(x["normalized_speed"].mean(), 4))
    display(
        x[obs_cols]
        .sort_values("normalized_speed")
        .head(10)
    )


== resnet34_cifar100_bs128_20e_1gpu.yaml
count: 16
mean normalized speed: 0.8165


,spec_key,partner_spec_key,normalized_speed,partner_runtime_ratio,pairwise_source_csv
350,resnet34_cifar100_bs128_20e_1gpu.yaml,vgg16_imagenet_bs128_1gpu.yaml,0.428834,0.893286,evaluation/lucid/results/pairwise_results.csv
356,resnet34_cifar100_bs128_20e_1gpu.yaml,xception_imagenet_bs128_1gpu.yaml,0.472152,1.059387,evaluation/lucid/results/pairwise_results.csv
360,resnet34_cifar100_bs128_20e_1gpu.yaml,xception_imagenet_bs64_1gpu.yaml,0.640285,0.854119,evaluation/lucid/results/pairwise_results.csv
344,resnet34_cifar100_bs128_20e_1gpu.yaml,resnet50_imagenet_bs128_1gpu.yaml,0.643708,1.029248,evaluation/lucid/results/pairwise_results.csv
179,resnet34_cifar100_bs128_20e_1gpu.yaml,efficientnet_imagenet_bs128_1gpu.yaml,0.754464,1.088344,evaluation/lucid/results/pairwise_results.csv
225,resnet34_cifar100_bs128_20e_1gpu.yaml,llama3_width_8layer_wiki_bs1_1gpu.yaml,0.776864,1.286817,evaluation/lucid/results/pairwise_results.csv
283,resnet34_cifar100_bs128_20e_1gpu.yaml,mobilenet_imagenet_bs128_1gpu.yaml,0.823690,1.089564,evaluation/lucid/results/pairwise_results.csv
346,resnet34_cifar100_bs128_20e_1gpu.yaml,resnet50_imagenet_bs64_1gpu.yaml,0.839187,0.854175,evaluation/lucid/results/pairwise_results.csv
348,resnet34_cifar100_bs128_20e_1gpu.yaml,unet_voc_1gpu_10e_1gpu.yaml,0.897578,1.115383,evaluation/lucid/results/pairwise_results.csv
123,resnet34_cifar100_bs128_20e_1gpu.yaml,efficientnet_cifar100_bs128_20e_1gpu.yaml,0.929133,1.481388,evaluation/lucid/results/pairwise_results.csv



== resnet34_cifar100_bs64_20e_1gpu.yaml
count: 6
mean normalized speed: 0.9886


,spec_key,partner_spec_key,normalized_speed,partner_runtime_ratio,pairwise_source_csv
76,resnet34_cifar100_bs64_20e_1gpu.yaml,unet_voc_1gpu.yaml,0.931408,4.725833,evaluation/lucid/results/pairwise_results.csv
49,resnet34_cifar100_bs64_20e_1gpu.yaml,mobilenet_cifar100_bs64_20e_1gpu.yaml,1.000000,1.201672,evaluation/lucid/results/pairwise_results.csv
65,resnet34_cifar100_bs64_20e_1gpu.yaml,resnet18_cifar100_bs32_20e_1gpu.yaml,1.000000,1.239521,evaluation/lucid/results/pairwise_results.csv
127,resnet34_cifar100_bs64_20e_1gpu.yaml,efficientnet_cifar100_bs128_20e_1gpu.yaml,1.000000,0.839494,evaluation/lucid/results/pairwise_results.csv
455,resnet34_cifar100_bs64_20e_1gpu.yaml,mobilenet_cifar100_bs64_20e_1gpu.yaml,1.000000,1.347517,evaluation/lucid/results/tiny_medium_calibrati...
478,resnet34_cifar100_bs64_20e_1gpu.yaml,efficientnet_cifar100_bs128_20e_1gpu.yaml,1.000000,0.849700,evaluation/lucid/results/tiny_medium_calibrati...



== mobilenet_cifar100_bs128_20e_1gpu.yaml
count: 17
mean normalized speed: 0.8928


,spec_key,partner_spec_key,normalized_speed,partner_runtime_ratio,pairwise_source_csv
248,mobilenet_cifar100_bs128_20e_1gpu.yaml,vgg16_imagenet_bs128_1gpu.yaml,0.486047,0.860699,evaluation/lucid/results/pairwise_results.csv
254,mobilenet_cifar100_bs128_20e_1gpu.yaml,xception_imagenet_bs128_1gpu.yaml,0.601805,1.127501,evaluation/lucid/results/pairwise_results.csv
258,mobilenet_cifar100_bs128_20e_1gpu.yaml,xception_imagenet_bs64_1gpu.yaml,0.784999,0.875539,evaluation/lucid/results/pairwise_results.csv
242,mobilenet_cifar100_bs128_20e_1gpu.yaml,resnet50_imagenet_bs128_1gpu.yaml,0.825524,1.108245,evaluation/lucid/results/pairwise_results.csv
450,mobilenet_cifar100_bs128_20e_1gpu.yaml,resnet18_cifar100_bs64_20e_1gpu.yaml,0.872934,0.928998,evaluation/lucid/results/tiny_medium_calibrati...
169,mobilenet_cifar100_bs128_20e_1gpu.yaml,efficientnet_imagenet_bs128_1gpu.yaml,0.889754,1.059287,evaluation/lucid/results/pairwise_results.csv
221,mobilenet_cifar100_bs128_20e_1gpu.yaml,llama3_width_8layer_wiki_bs1_1gpu.yaml,0.892565,1.228960,evaluation/lucid/results/pairwise_results.csv
232,mobilenet_cifar100_bs128_20e_1gpu.yaml,mobilenet_imagenet_bs128_1gpu.yaml,0.892899,1.007442,evaluation/lucid/results/pairwise_results.csv
230,mobilenet_cifar100_bs128_20e_1gpu.yaml,mobilenet_cifar100_bs64_20e_1gpu.yaml,0.940363,1.819336,evaluation/lucid/results/pairwise_results.csv
246,mobilenet_cifar100_bs128_20e_1gpu.yaml,unet_voc_1gpu_10e_1gpu.yaml,0.991505,1.007110,evaluation/lucid/results/pairwise_results.csv



== mobilenet_cifar100_bs64_20e_1gpu.yaml
count: 6
mean normalized speed: 0.9991


,spec_key,partner_spec_key,normalized_speed,partner_runtime_ratio,pairwise_source_csv
48,mobilenet_cifar100_bs64_20e_1gpu.yaml,resnet34_cifar100_bs64_20e_1gpu.yaml,0.994884,0.832174,evaluation/lucid/results/pairwise_results.csv
46,mobilenet_cifar100_bs64_20e_1gpu.yaml,resnet18_cifar100_bs32_20e_1gpu.yaml,1.000000,1.186192,evaluation/lucid/results/pairwise_results.csv
52,mobilenet_cifar100_bs64_20e_1gpu.yaml,unet_voc_1gpu.yaml,1.000000,4.792756,evaluation/lucid/results/pairwise_results.csv
163,mobilenet_cifar100_bs64_20e_1gpu.yaml,efficientnet_cifar100_bs32_20e_1gpu.yaml,1.000000,2.707814,evaluation/lucid/results/pairwise_results.csv
466,mobilenet_cifar100_bs64_20e_1gpu.yaml,efficientnet_cifar100_bs32_10e_1gpu.yaml,1.000000,1.266969,evaluation/lucid/results/tiny_medium_calibrati...
476,mobilenet_cifar100_bs64_20e_1gpu.yaml,resnet34_cifar100_bs32_15e_1gpu.yaml,1.000000,1.277216,evaluation/lucid/results/tiny_medium_calibrati...


Lucid already recognizes that colocation slowdown depends on job pairs and uses pairwise measurements to construct sharing scores. However, this design necessarily compresses a distribution of pairwise interactions into a single per-job class. Our measurements show that this compression can be sensitive to the partner set and workload configuration, motivating AEGIS’s runtime-aware placement.